# 12 — Matched Model Comparisons and Uncertainty

## Objective
Compare untouched, SFT (Supervised Fine-Tuning), DPO (Direct Preference Optimization), and GRPO (Group Relative Policy Optimization) variants with **the same validation papers, prompts, decoding controls, and scoring logic**. Reject comparisons involving mismatched model IDs, incomplete source provenance, or different evaluation subsets.

## Experimental limitations
DPO preference supervision is drawn from Med-REFL-DPO, while SFT and GRPO use PubMedQA-derived supervision. Consequently, a change between branches is not solely an isolated optimizer effect. Record training datasets and model histories alongside scores.

### Source, split, and checkpoint locations

In [1]:
# Locate the research workspace from the repository root or the notebooks folder.
from pathlib import Path
cwd = Path.cwd().resolve()
if (cwd / "exploration" / "notebooks").is_dir():
    RESEARCH = cwd / "exploration"
elif cwd.name == "notebooks" and cwd.parent.name == "exploration":
    RESEARCH = cwd.parent
elif cwd.name == "exploration" and (cwd / "notebooks").is_dir():
    RESEARCH = cwd
else:
    raise FileNotFoundError("Open Jupyter from the repository root, exploration/, or exploration/notebooks/.")

RUNS = RESEARCH / "runs"
RUNS.mkdir(parents=True, exist_ok=True)
print("Research workspace:", RESEARCH)
print("Experiment outputs:", RUNS)

Research workspace: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration
Experiment outputs: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration\runs


In [2]:
# PubMedQA dataset protocol: the same source, group assignment and prompt data
# must be used in notebooks 03, 04, 05, 06, 11, 12 and 18.
import hashlib
import json
import os
from collections import Counter

DATASET_NAME = "qiaojin/PubMedQA"
DATASET_CONFIG = "pqa_labeled"      # Explicitly labeled PubMedQA questions.
SEED = 42
LABELS = ("yes", "no", "maybe")
MANIFEST = RESEARCH / "data" / "pubmedqa_split_manifest.json"
MANIFEST.parent.mkdir(parents=True, exist_ok=True)


def paper_id(row):
    """Stable publication ID used to prevent paper-level train/evaluation overlap."""
    value = str(row.get("pubid", "")).strip()
    if not value:
        raise ValueError("Missing PubMedQA publication ID; cannot split safely.")
    return value


def evidence_text(row):
    """Extract abstract sentences from PubMedQA's nested context object."""
    context = row.get("context") or {}
    sentences = context.get("contexts", []) if isinstance(context, dict) else []
    if not isinstance(sentences, list):
        raise TypeError("Expected context['contexts'] to be a list of abstract sentences.")
    return "\n".join(sentence.strip() for sentence in sentences if isinstance(sentence, str))


def assign_split(identifier):
    """Deterministically assign ALL rows with one publication ID to the same split."""
    digest = hashlib.sha256(f"{SEED}:{identifier}".encode("utf-8")).digest()
    fraction = int.from_bytes(digest[:8], "big") / 2**64
    if fraction < 0.80:
        return "train"
    if fraction < 0.90:
        return "validation"
    return "test"


def source_fingerprint(rows):
    """Hash canonical source fields to detect modifications between experiments."""
    canonical = sorted(
        (paper_id(row), str(row["question"]), evidence_text(row), str(row["final_decision"]).lower())
        for row in rows
    )
    payload = json.dumps(canonical, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def prepare_partitions(rows, create_manifest=False):
    """Check labels, record stable splits and require identical upstream source data."""
    records = list(rows)
    identifiers = [paper_id(row) for row in records]
    if len(identifiers) != len(set(identifiers)):
        raise ValueError("Repeated publication IDs: review source records before evaluation.")
    if any(str(row["final_decision"]).lower() not in LABELS for row in records):
        raise ValueError("Unexpected label; expected one of yes/no/maybe.")
    if any(not evidence_text(row) for row in records):
        raise ValueError("Empty evidence context; review source-data schema.")

    partitions = {name: [] for name in ("train", "validation", "test")}
    for row in records:
        partitions[assign_split(paper_id(row))].append(row)
    metadata = {
        "dataset": DATASET_NAME,
        "config": DATASET_CONFIG,
        "split_algorithm": "SHA256(seed:pubid), buckets 0-0.8/0.8-0.9/0.9-1.0",
        "seed": SEED,
        "fingerprint": source_fingerprint(records),
        "ids": {name: sorted(paper_id(row) for row in subset) for name, subset in partitions.items()},
    }
    if MANIFEST.exists():
        previous = json.loads(MANIFEST.read_text(encoding="utf-8"))
        if previous != metadata:
            raise RuntimeError("Dataset differs from saved split manifest. Do not silently overwrite it; inspect data revision and protocol.")
    elif create_manifest:
        MANIFEST.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
        print("Created reproducible split manifest:", MANIFEST)
    else:
        raise FileNotFoundError("Split manifest missing. Run notebook 03 with RUN_PUBLIC_DATA=1 first.")

    for name, subset in partitions.items():
        print(f"{name}: n={len(subset)}, labels={dict(Counter(str(r['final_decision']).lower() for r in subset))}")
    if min(len(p) for p in partitions.values()) == 0:
        raise ValueError("At least one split is empty; check dataset size and split rules.")
    return partitions

### Shared prompt and metric definitions

In [3]:
SYSTEM_INSTRUCTION = (
    "Answer the biomedical question using ONLY the supplied abstract evidence. "
    "Choose exactly one decision label: yes, no, or maybe. "
    "Return the label alone, without an explanation. "
    "Use maybe when the evidence does not establish yes or no."
)


def question_messages(row):
    """Use one consistent role-aware input protocol across model variants."""
    full_text = evidence_text(row)
    if not full_text:
        raise ValueError("Question has no evidence passage.")
    # Keep both the abstract introduction and conclusion; the conclusion often
    # carries the decision evidence. Apply the same limit in training/evaluation.
    if len(full_text) > 1600:
        text = full_text[:950] + "\n[Middle of abstract omitted for length]\n" + full_text[-600:]
    else:
        text = full_text
    return [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": f"Question: {row['question']}\n\nAbstract evidence:\n{text}"},
    ]


def parse_decision(generated_text):
    """Parse an exact one-label answer; preserve raw outputs when parsing fails."""
    import re
    text = str(generated_text).strip()
    match = re.fullmatch(r"(?i)(yes|no|maybe)[.\s]*", text)
    return match.group(1).lower() if match else "invalid"

In [4]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score


def assess_predictions(frame):
    """Report output validity, class balance, and independent majority baseline."""
    if frame.empty:
        raise ValueError("No predictions provided for scoring.")
    y_true = frame["label"].astype(str).str.lower()
    y_pred = frame["prediction"].astype(str).str.lower()
    if not set(y_true).issubset(set(LABELS)):
        raise ValueError("Reference labels must be yes/no/maybe.")
    counts = y_true.value_counts()
    majority_label = counts.idxmax()
    summary = {
        "n": len(frame),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=list(LABELS), average="macro", zero_division=0)),
        "valid_format": float(y_pred.isin(LABELS).mean()),
        "majority_label": majority_label,
        "majority_accuracy": float((y_true == majority_label).mean()),
        "uniform_random_accuracy": 1 / len(LABELS),
    }
    return summary


def show_diagnostics(frame):
    """Provide evidence to distinguish formatting errors from actual classification errors."""
    metrics = assess_predictions(frame)
    print("Sample size:", metrics["n"])
    print("Reference distribution:", frame["label"].value_counts().to_dict())
    print("Predicted distribution:", frame["prediction"].value_counts().to_dict())
    print("Summary:", metrics)
    print("Confusion matrix (rows=reference, columns=prediction):")
    labels = list(LABELS) + ["invalid"]
    print(pd.DataFrame(confusion_matrix(frame["label"], frame["prediction"], labels=labels),
                       index=labels, columns=labels))
    mistakes = frame.loc[frame["prediction"] != frame["label"]]
    inspect_columns = [c for c in ("id", "label", "prediction", "raw", "input_token_count") if c in mistakes]
    if not mistakes.empty:
        print("Error samples (inspect generation and label handling):")
        display(mistakes[inspect_columns].head(8))
    if metrics["n"] < 100:
        print("CAUTION: A sample of fewer than 100 records gives an imprecise performance estimate.")
    if metrics["accuracy"] <= metrics["majority_accuracy"]:
        print("CAUTION: The model does not beat the validation majority-class baseline.")
    return metrics

In [5]:
# Use the tokenizer's instruction-chat format: raw plain-text prompts are not equivalent.
# Extract newly generated tokens only, and preserve the exact decoded text for auditing.
def generate_answer(model, tokenizer, row, max_input_tokens=768, max_output_tokens=8):
    import torch
    from transformers import GenerationConfig

    if tokenizer.chat_template is None:
        raise ValueError("Model tokenizer does not provide the expected chat template.")
    chat_prompt = tokenizer.apply_chat_template(
        question_messages(row), tokenize=False, add_generation_prompt=True
    )
    encoded = tokenizer(
        chat_prompt,
        return_tensors="pt",
        truncation=False,
        add_special_tokens=False,  # Chat template already contains special tokens.
    )
    input_length = int(encoded["input_ids"].shape[-1])
    if input_length > max_input_tokens:
        raise ValueError(
            f"Paper {paper_id(row)} exceeds the input token budget "
            f"({input_length} > {max_input_tokens}); revise the shared evidence budget. "
            "Do not truncate the assistant generation marker."
        )
    encoded = {key: value.to(model.device) for key, value in encoded.items()}

    # Ignore sampling values embedded in a model repository's generation_config.json.
    # A freshly constructed config keeps this run deterministic and avoids warnings
    # about sampling-only flags when do_sample=False.
    config = GenerationConfig(
        do_sample=False,
        max_new_tokens=max_output_tokens,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    with torch.inference_mode():
        sequence = model.generate(**encoded, generation_config=config)
    response = tokenizer.decode(sequence[0, input_length:], skip_special_tokens=True)
    return {"raw": response, "prediction": parse_decision(response), "input_token_count": input_length}

### Opt-in independent inference for all available model variants
`RUN_MODEL_EVAL=1` evaluates every available variant with one fixed benchmark. Only variants with an explicitly saved training manifest are accepted. This cell may require significant memory and time.

In [6]:
import gc
import os
import json
import time

if os.getenv("RUN_MODEL_EVAL", "0") == "1":
    import torch
    import transformers
    import datasets
    from datasets import Dataset
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftConfig, PeftModel

    model_id = os.getenv("MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
    prior = json.loads((RUNS / "predictions_base_validation_manifest.json").read_text())
    revision = prior["model_revision"]
    if model_id != prior["model_id"]:
        raise ValueError("Requested base model differs from the frozen baseline evaluation.")
    cached = list(__import__("pathlib").Path(datasets.config.HF_DATASETS_CACHE).glob(
        "qiaojin___pub_med_qa/pqa_labeled/*/*/pub_med_qa-train.arrow"))
    if len(cached) != 1:
        raise FileNotFoundError(f"Expected one cached PubMedQA Arrow source, found {len(cached)}")
    source = Dataset.from_file(str(cached[0]))
    partitions = prepare_partitions(source, create_manifest=False)
    validation = sorted(partitions["validation"], key=paper_id)
    max_items = int(os.getenv("EVAL_MAX_ITEMS", "0"))
    if max_items > 0:
        validation = validation[:max_items]
        print("DIAGNOSTIC SUBSET: this is not a full-validation benchmark.")
    fingerprint = json.loads(MANIFEST.read_text())["fingerprint"]
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision, local_files_only=True)
    variants = {"base": None, "sft": RUNS / "sft_adapter",
                "dpo": RUNS / "dpo_adapter", "grpo": RUNS / "grpo_adapter"}
    output_dir = RUNS / "comparison_12"
    output_dir.mkdir(exist_ok=True)
    expected_ids = [paper_id(row) for row in validation]
    for variant, adapter_dir in variants.items():
        training = None
        if adapter_dir is not None:
            training = json.loads((adapter_dir / "run_manifest.json").read_text())
            config = PeftConfig.from_pretrained(str(adapter_dir))
            if (training["model_id"] != model_id or training["model_revision"] != revision
                    or training["dataset_fingerprint"] != fingerprint
                    or config.base_model_name_or_path != model_id):
                raise ValueError(f"{variant} adapter lineage differs from frozen source/base.")
        torch.manual_seed(SEED)
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        started = time.perf_counter()
        model = AutoModelForCausalLM.from_pretrained(
            model_id, revision=revision, local_files_only=True, device_map="auto", torch_dtype="auto")
        if getattr(model.config, "_commit_hash", None) != revision:
            raise ValueError("Loaded base checkpoint revision differs from frozen baseline.")
        if adapter_dir is not None:
            model = PeftModel.from_pretrained(model, str(adapter_dir))
        model.eval()
        predictions = []
        for index, row in enumerate(validation, 1):
            predictions.append({"id": paper_id(row), "label": str(row["final_decision"]).lower(),
                                "question": str(row["question"]), "model_id": model_id,
                                **generate_answer(model, tokenizer, row)})
            if index % 40 == 0 or index == len(validation):
                print(f"{variant}: {index}/{len(validation)}")
        frame = pd.DataFrame(predictions)
        if frame["id"].tolist() != expected_ids:
            raise ValueError("Validation IDs changed during inference.")
        if variant in ("base", "sft") and max_items == 0:
            earlier = pd.read_csv(RUNS / f"predictions_{variant}_validation.csv", dtype={"id": str})
            columns = ["id", "label", "question", "raw", "prediction", "input_token_count", "model_id"]
            if not frame[columns].equals(earlier[columns]):
                raise ValueError(f"{variant} predictions differ from the preserved earlier run.")
            print(f"{variant}: exact row-level agreement with earlier run")
        print("Completed variant:", variant)
        show_diagnostics(frame)
        csv_path = output_dir / f"predictions_{variant}_validation.csv"
        frame.to_csv(csv_path, index=False)
        protocol = {"model_variant": variant, "model_id": model_id, "model_revision": revision,
                    "dataset_fingerprint": fingerprint, "source_partition": "validation",
                    "evaluated_ids": expected_ids, "decoding": "chat_template/greedy/max_new_tokens=8",
                    "max_items": max_items, "seed": SEED, "device": str(model.device),
                    "model_dtype": str(model.dtype), "torch_version": torch.__version__,
                    "transformers_version": transformers.__version__,
                    "adapter_method": training.get("method") if training else None,
                    "elapsed_seconds": round(time.perf_counter() - started, 3)}
        (output_dir / f"predictions_{variant}_validation_manifest.json").write_text(
            json.dumps(protocol, indent=2) + "\n", encoding="utf-8")
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
else:
    print("Inference disabled. Set RUN_MODEL_EVAL=1 to evaluate real checkpoints.")

train: n=775, labels={'yes': 433, 'no': 255, 'maybe': 87}
validation: n=121, labels={'no': 40, 'yes': 64, 'maybe': 17}
test: n=104, labels={'yes': 55, 'no': 43, 'maybe': 6}


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


C:\Users\weiju\anaconda3\envs\torch\lib\site-packages\transformers\integrations\sdpa_attention.py:92: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


base: 40/121


...

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

grpo: 40/121


grpo: 80/121


grpo: 120/121
grpo: 121/121
Completed variant: grpo
Sample size: 121
Reference distribution: {'yes': 64, 'no': 40, 'maybe': 17}
Predicted distribution: {'yes': 103, 'no': 18}
Summary: {'n': 121, 'accuracy': 0.628099173553719, 'macro_f1': 0.4084245302498451, 'valid_format': 1.0, 'majority_label': 'yes', 'majority_accuracy': 0.5289256198347108, 'uniform_random_accuracy': 0.3333333333333333}
Confusion matrix (rows=reference, columns=prediction):
         yes  no  maybe  invalid
yes       62   2      0        0
no        26  14      0        0
maybe     15   2      0        0
invalid    0   0      0        0
Error samples (inspect generation and label handling):


,id,label,prediction,raw,input_token_count
3,11138995,no,yes,yes,338
4,11458136,maybe,no,no,314
6,11713724,no,yes,yes,433
7,11926574,no,yes,yes,436
9,12068831,no,yes,yes,220
10,12238307,no,yes,yes,256
13,12805495,maybe,yes,yes,383
15,12920330,maybe,yes,yes,283


### Immediate interpretation of model outputs

The four raw inference passes used the same 121 validation examples. Base and SFT exactly reproduced their earlier row-level files. DPO and GRPO each produced a saved prediction CSV and protocol manifest. All trained variants were valid-format on every case but emitted no maybe decisions; the paired comparisons and class-wise matrices below determine whether any apparent gain is distinguishable from a few changed cases.

### Compare saved, protocol-compatible validation results and estimate uncertainty

In [7]:
from pathlib import Path

comparison_dir = RUNS / "comparison_12"
records, compatible = [], []
reference_protocol = None
for variant in ("base", "sft", "dpo", "grpo"):
    csv_path = comparison_dir / f"predictions_{variant}_validation.csv"
    protocol_path = comparison_dir / f"predictions_{variant}_validation_manifest.json"
    if not csv_path.exists() or not protocol_path.exists():
        raise FileNotFoundError(f"Missing comparison output for {variant}")
    protocol = json.loads(protocol_path.read_text())
    keys = ("model_id", "model_revision", "dataset_fingerprint", "source_partition",
            "evaluated_ids", "decoding", "max_items", "seed", "model_dtype",
            "torch_version", "transformers_version")
    compatibility = {key: protocol[key] for key in keys}
    if reference_protocol is None:
        reference_protocol = compatibility
    if compatibility != reference_protocol or protocol["model_variant"] != variant:
        raise ValueError(f"Incompatible evaluation protocol for {variant}")
    frame = pd.read_csv(csv_path, dtype={"id": str})
    if frame["id"].tolist() != protocol["evaluated_ids"] or frame["id"].duplicated().any():
        raise ValueError(f"CSV and manifest identifiers mismatch for {variant}")
    records.append({"variant": variant, **assess_predictions(frame)})
    compatible.append((variant, frame))

display(pd.DataFrame(records))
reference = compatible[0][1]
for variant, frame in compatible[1:]:
    for column in ("id", "label", "question", "input_token_count"):
        if not frame[column].equals(reference[column]):
            raise ValueError(f"Unmatched {column} for {variant}")

# Resample publication IDs together so every model is assessed on the same cases.
for reference_name in ("base", "sft"):
    ref = dict(compatible)[reference_name]
    for variant, frame in compatible:
        if variant == reference_name:
            continue
        differences = (frame["prediction"].eq(frame["label"]).to_numpy(dtype=float)
                       - ref["prediction"].eq(ref["label"]).to_numpy(dtype=float))
        rng = np.random.default_rng(SEED)
        bootstrap = np.array([differences[rng.integers(len(differences), size=len(differences))].mean()
                              for _ in range(10000)])
        print(f"{variant} minus {reference_name}: accuracy difference={differences.mean():+.3f}; "
              f"paired bootstrap 95% interval={np.quantile(bootstrap, [0.025, 0.975])}")

for variant, frame in compatible:
    print(f"{variant} confusion matrix (rows: reference; columns: yes/no/maybe/invalid)")
    display(pd.crosstab(frame["label"], frame["prediction"]).reindex(
        index=LABELS, columns=(*LABELS, "invalid"), fill_value=0))

majority = reference["label"].value_counts().max() / len(reference)
print(f"Validation majority-class accuracy: {majority:.3f}; n={len(reference)}")

# Exact paired sign test exposes when an apparent gain rests on very few disagreements.
from scipy.stats import binomtest
sft = dict(compatible)["sft"]
for variant in ("dpo", "grpo"):
    candidate = dict(compatible)[variant]
    gained = int((candidate["prediction"].eq(candidate["label"]) &
                  ~sft["prediction"].eq(sft["label"])).sum())
    lost = int((~candidate["prediction"].eq(candidate["label"]) &
                sft["prediction"].eq(sft["label"])).sum())
    p_value = binomtest(gained, gained + lost, 0.5).pvalue if gained + lost else 1.0
    print(f"{variant} versus SFT: gained={gained}, lost={lost}, exact paired p={p_value:.4f}")

,variant,n,accuracy,macro_f1,valid_format,majority_label,majority_accuracy,uniform_random_accuracy
0,base,121,0.157025,0.115079,1.0,yes,0.528926,0.333333
1,sft,121,0.603306,0.421552,1.0,yes,0.528926,0.333333
2,dpo,121,0.636364,0.447414,1.0,yes,0.528926,0.333333
3,grpo,121,0.628099,0.408425,1.0,yes,0.528926,0.333333


sft minus base: accuracy difference=+0.446; paired bootstrap 95% interval=[0.31404959 0.57024793]
dpo minus base: accuracy difference=+0.479; paired bootstrap 95% interval=[0.34710744 0.60330579]
grpo minus base: accuracy difference=+0.471; paired bootstrap 95% interval=[0.33884298 0.59504132]
base minus sft: accuracy difference=-0.446; paired bootstrap 95% interval=[-0.57024793 -0.31404959]


dpo minus sft: accuracy difference=+0.033; paired bootstrap 95% interval=[0.00826446 0.0661157 ]
grpo minus sft: accuracy difference=+0.025; paired bootstrap 95% interval=[-0.04958678  0.09917355]
base confusion matrix (rows: reference; columns: yes/no/maybe/invalid)


prediction,yes,no,maybe,invalid
label,,,,
yes,0,0,64,0
no,0,2,38,0
maybe,0,0,17,0


sft confusion matrix (rows: reference; columns: yes/no/maybe/invalid)


...

dpo confusion matrix (rows: reference; columns: yes/no/maybe/invalid)


prediction,yes,no,maybe,invalid
label,,,,
yes,52,12,0,0
no,15,25,0,0
maybe,14,3,0,0


grpo confusion matrix (rows: reference; columns: yes/no/maybe/invalid)


prediction,yes,no,maybe,invalid
label,,,,
yes,62,2,0,0
no,26,14,0,0
maybe,15,2,0,0


Validation majority-class accuracy: 0.529; n=121
dpo versus SFT: gained=4, lost=0, exact paired p=0.1250
grpo versus SFT: gained=12, lost=9, exact paired p=0.6636


### Results

On identical 121-question validation rows, base/SFT/DPO/GRPO scored 19/73/77/76 correct, with macro-F1 0.115/0.422/0.447/0.408. The majority-yes rule scored 64/121. DPO gained four SFT errors and lost none (exact two-sided paired p = 0.125). GRPO gained 12 and lost nine (p about 0.664). No trained adapter predicted maybe for the 17 true-maybe cases.

### Interpretation

SFT clearly exceeded the weak base on this protocol. DPO and GRPO numerical increments over SFT are too uncertain to establish a robust winner; GRPO's macro-F1 was below SFT's.

### Limitations

The validation set has been examined repeatedly. DPO and GRPO used different training sources and budgets, and the paired tests are exploratory, not multiplicity-adjusted confirmatory evidence.

### Key Observations

Freeze a candidate and analysis plan independently before using the sealed 104-record final test.


## Interpretation
A confidence interval containing zero is consistent with no detectable change at the available sample size; it is not evidence that methods are equivalent. Compare macro-F1, error types, answer validity, and per-class confusion matrices in addition to raw accuracy. Model improvements must also be evaluated on a sealed test set after model selection.

## Q&A
**Question:** Why exclude an earlier CSV even if its accuracy looks favorable?  
**Answer:** Without an identical dataset fingerprint, model ID, record set and decoding protocol, the result is not a controlled comparison. A favorable mismatched number is not meaningful evidence of an algorithmic improvement.